In [2]:
import pandas as pd
import json
from pathlib import Path

state_file = Path("../state/watermark.json")
state_file.parent.mkdir(parents=True, exist_ok=True)
fact_file = Path("../data/enrich/fact_fuel_transactions.csv")
LOOKBACK_DAYS = 2

def read_watermark():
    if state_file.exists():
        return pd.to_datetime(json.loads(state_file.read_text())["last_loaded"])
    return pd.to_datetime("2025-12-31")

def write_watermark(ts):
    state_file.write_text(json.dumps({"last_loaded": str(ts.date())}))

def load_increment(new_txns):
    wm = read_watermark()
    start = wm - pd.Timedelta(days=LOOKBACK_DAYS)
    batch = new_txns[new_txns["txn_date"] > start]

    fact = pd.read_csv(fact_file, parse_dates=["txn_date"])
    fact = fact[fact["txn_date"] <= start]
    fact = pd.concat([fact, batch]).drop_duplicates(subset="txn_id", keep="last")
    fact.to_csv(fact_file, index=False)

    write_watermark(batch["txn_date"].max() if len(batch) else wm)
    print(f"Watermark {wm.date()} -> loaded {len(batch)} rows, fact now {len(fact)}")

In [3]:
enriched = pd.read_csv(fact_file, parse_dates=["txn_date"])

load_increment(enriched)
load_increment(enriched)
load_increment(enriched)

Watermark 2025-12-31 -> loaded 5000 rows, fact now 5000
Watermark 2026-10-27 -> loaded 20 rows, fact now 5000
Watermark 2026-10-27 -> loaded 20 rows, fact now 5000


In [5]:
fact = pd.read_csv(fact_file, parse_dates=["txn_date"])

daily = fact.groupby([fact["txn_date"].dt.date, "station"]).size().unstack(fill_value=0)

all_days = pd.date_range(fact["txn_date"].min(), fact["txn_date"].max()).date
daily = daily.reindex(all_days, fill_value=0)

mask = daily.eq(0).stack()
missing = mask[mask]
print(len(missing), "station-days with zero transactions")
print(missing.head(10))

22 station-days with zero transactions
txn_date    station     
2026-01-10  Ipoh            True
2026-01-24  Penang          True
2026-02-02  Johor Bahru     True
2026-02-03  Ipoh            True
2026-02-11  Penang          True
2026-02-19  Kuala Lumpur    True
2026-03-10  Penang          True
2026-03-18  Penang          True
2026-03-20  Johor Bahru     True
2026-03-21  Kuala Lumpur    True
dtype: bool
